In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.silver.silver_production_plan AS
WITH exploded_plan AS (
  SELECT 
    record_type,
    upper(trim(line_code)) AS line_code,
    trim(line_name) AS line_name,
    
    -- Rozbicie tablicy komórek na pojedyncze rekordy
    explode_outer(assigned_cells) AS cell_struct,
    
    CAST(start_datetime AS TIMESTAMP) AS start_datetime,
    CAST(CAST(start_datetime AS TIMESTAMP) AS DATE) AS plan_date,
    CAST(end_datetime AS TIMESTAMP) AS end_datetime,
    
    plan_status,
    planned_capacity_pct,
    remarks,
    source,
    _source_file,
    _ingested_at AS _bronze_ingested_at,
    current_timestamp() AS _silver_ingested_at
  FROM data_warehouse_factory.bronze.bronze_production_plan
)
SELECT 
  -- Klucz sztuczny per komórka i okno czasowe
  md5(concat_ws('||', line_code, coalesce(cell_struct.cell_code, 'NO_CELL'), cast(start_datetime as string), _source_file)) AS plan_key,
  
  record_type,
  line_code,
  line_name,
  
  -- Wyciągnięcie kodu i nazwy ze struktury
  upper(trim(cell_struct.cell_code)) AS cell_code,
  trim(cell_struct.cell_name) AS cell_name,
  
  plan_date,
  start_datetime,
  end_datetime,
  
  -- Wyliczenie czasu trwania w godzinach
  ROUND(timestampdiff(MINUTE, start_datetime, end_datetime) / 60.0, 2) AS duration_hours,
  
  plan_status,
  planned_capacity_pct,
  remarks,
  source,
  _source_file,
  _bronze_ingested_at,
  _silver_ingested_at
FROM exploded_plan;